# Step 1 of 3 — Data Cleaning
**Pipeline:** `01_data_cleaning` → `02_sql_analysis` → `03_eda`

**Reads:** raw CSV (`config.RAW_CSV`)
**Writes:**
- `output/customer_behavior_clean.csv` → used by Step 2 (SQL) and Step 3 (EDA)
- `output/customer_behavior_treated.csv` → outliers capped + log columns, used by Step 3 (EDA)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import config

sns.set_theme(style="whitegrid")

## 1. Load dataset

In [ ]:
df = pd.read_csv(config.RAW_CSV)
print("Loaded:", config.RAW_CSV.name)
df

## 2. First look at the data

In [ ]:
print("Dataset Shape:", df.shape)

print("\nFirst 5 Rows:")
display(df.head())

print("\nRandom 5 Rows:")
display(df.sample(5, random_state=42))

print("\nLast 5 Rows:")
display(df.tail())

In [ ]:
print("Dataset Information:")
df.info()

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:", df.duplicated().sum())

In [ ]:
print("Statistical Summary:")
display(df.describe())

print("\nCorrelation:")
display(df.corr(numeric_only=True))

## 3. Standardise column names, text and data types

In [ ]:
# Lower-case column names (matches the SQL table), strip stray spaces in text columns
df.columns = df.columns.str.strip().str.lower()

text_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
for c in text_cols:
    df[c] = df[c].astype("string").str.strip()

# Correct data types
df["visit_date"] = pd.to_datetime(df["visit_date"], errors="coerce")
df["session_id"] = df["session_id"].astype("string")

print("Invalid / unparseable dates:", df["visit_date"].isna().sum())
print()
print(df.dtypes)

In [ ]:
numerical_cols = [
    "customer_age", "unit_price_inr", "quantity", "discount_percent",
    "revenue_inr", "rating", "session_duration_min", "pages_viewed",
]
categorical_cols = [
    "session_id", "customer_id", "visit_date", "city", "state",
    "customer_type", "device_type", "marketing_channel", "product_category",
    "purchased", "cart_abandoned", "payment_method",
]
print("Numerical columns :", len(numerical_cols), numerical_cols)
print("Categorical columns:", len(categorical_cols), categorical_cols)

plt.figure(figsize=(7, 5))
plt.bar(["Numerical", "Categorical"], [len(numerical_cols), len(categorical_cols)])
plt.title("Numerical vs Categorical Columns")
plt.ylabel("Number of Columns")
plt.tight_layout()
plt.show()

## 4. Missing values

In [ ]:
print((df.isnull().mean() * 100).round(2).rename("missing_%"))

# Are the missing ratings simply the sessions with no purchase?
print("\nMissing ratings        :", df["rating"].isna().sum())
print("Sessions not purchased :", (df["purchased"] == 0).sum())
print("Missing rating AND not purchased:", (df["rating"].isna() & (df["purchased"] == 0)).sum())

In [ ]:
# Keep a flag so the information "customer never rated" is not lost, then fill with median
df["has_rating"] = df["rating"].notna().astype(int)
df["rating"] = df["rating"].fillna(df["rating"].median())

print("Missing values left:", df.isnull().sum().sum())

## 5. Duplicates and invalid values

In [ ]:
before = len(df)
df = df.drop_duplicates().drop_duplicates(subset="session_id").reset_index(drop=True)
print(f"Rows before: {before} | after removing duplicates: {len(df)}")

checks = {
    "age outside 0-120":        ((df["customer_age"] < 0) | (df["customer_age"] > 120)).sum(),
    "negative unit price":      (df["unit_price_inr"] < 0).sum(),
    "negative quantity":        (df["quantity"] < 0).sum(),
    "discount outside 0-100":   ((df["discount_percent"] < 0) | (df["discount_percent"] > 100)).sum(),
    "negative revenue":         (df["revenue_inr"] < 0).sum(),
    "purchased not 0/1":        (~df["purchased"].isin([0, 1])).sum(),
    "cart_abandoned not 0/1":   (~df["cart_abandoned"].isin([0, 1])).sum(),
    "revenue > 0 but not purchased": ((df["revenue_inr"] > 0) & (df["purchased"] == 0)).sum(),
}
print()
print(pd.Series(checks, name="invalid_rows"))

## 6. Add helper column and save the CLEAN file (→ Step 2 and Step 3)

In [ ]:
# Month column is used by the SQL "revenue by month" query
df["month"] = df["visit_date"].dt.strftime("%Y-%m")

df.to_csv(config.CLEAN_CSV, index=False)
print("Saved:", config.CLEAN_CSV)
print("Rows, columns:", df.shape)

## 7. Outlier treatment & skewness reduction
IQR capping + `log1p` transformation. The result is saved **separately** (`treated` file) because capping
changes real revenue values — totals/KPIs in SQL therefore use the *clean* file, not this one.

In [ ]:
numeric_cols = ["customer_age", "unit_price_inr", "quantity", "discount_percent", "revenue_inr"]

df_treated = df.copy()
for col in numeric_cols:
    df_treated[col] = pd.to_numeric(df_treated[col], errors="coerce")

print("SKEWNESS BEFORE TREATMENT")
print(df_treated[numeric_cols].skew().round(2))

# IQR capping
bounds = {}
for col in numeric_cols:
    q1, q3 = df_treated[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    bounds[col] = (lower, upper)
    n_out = ((df_treated[col] < lower) | (df_treated[col] > upper)).sum()
    df_treated[col] = df_treated[col].clip(lower=lower, upper=upper)
    print(f"{col:18s} capped {n_out:4d} values to [{lower:.2f}, {upper:.2f}]")

# log1p for still right-skewed, non-negative columns
log_cols = []
for col in numeric_cols:
    s = df_treated[col].dropna()
    if len(s) > 2 and s.skew() > 0.5 and (s >= 0).all():
        df_treated[col + "_log"] = np.log1p(df_treated[col])
        log_cols.append(col + "_log")

comparison = pd.DataFrame({
    "Original skew": df[numeric_cols].skew(),
    "After capping": df_treated[numeric_cols].skew(),
})
print("\nSKEWNESS COMPARISON")
display(comparison.round(2))
if log_cols:
    print("SKEWNESS AFTER LOG TRANSFORM")
    print(df_treated[log_cols].skew().round(2))

In [ ]:
for col in numeric_cols:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    sns.histplot(df[col].dropna(), kde=True, ax=axes[0]);          axes[0].set_title(f"{col}: Before Treatment")
    sns.histplot(df_treated[col].dropna(), kde=True, ax=axes[1]);  axes[1].set_title(f"{col}: After Capping")
    sns.boxplot(x=df_treated[col].dropna(), ax=axes[2]);           axes[2].set_title(f"{col}: Cleaned Boxplot")
    plt.tight_layout()
    plt.show()

In [ ]:
df_treated.to_csv(config.TREATED_CSV, index=False)
print("Saved:", config.TREATED_CSV)
print("Original rows:", len(df), "| Treated rows:", len(df_treated))
print("New log columns:", log_cols)
print("\n✔ Step 1 complete  →  now run 02_sql_analysis.ipynb")